In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

### Getting the Headcount for the data values by State

Top 5 states:

In [2]:
df = pd.read_csv("pollution_2000_2023.csv")
state_counts = df["State"].value_counts()

top_5_states = state_counts.head(5)
print(top_5_states)

State
California      201244
Pennsylvania     54142
Texas            38980
Arizona          25118
New York         22685
Name: count, dtype: int64


Understanading the Shape:

In [3]:
#creates a new clean copy of the dataset
df_clean = df.copy()

print(df_clean.shape)
print(df_clean.head())
print(df_clean.info())

(665414, 22)
   Unnamed: 0        Date                                  Address    State  \
0           0  2000-01-01  1645 E ROOSEVELT ST-CENTRAL PHOENIX STN  Arizona   
1           1  2000-01-02  1645 E ROOSEVELT ST-CENTRAL PHOENIX STN  Arizona   
2           2  2000-01-03  1645 E ROOSEVELT ST-CENTRAL PHOENIX STN  Arizona   
3           3  2000-01-04  1645 E ROOSEVELT ST-CENTRAL PHOENIX STN  Arizona   
4           4  2000-01-05  1645 E ROOSEVELT ST-CENTRAL PHOENIX STN  Arizona   

     County     City   O3 Mean  O3 1st Max Value  O3 1st Max Hour  O3 AQI  \
0  Maricopa  Phoenix  0.019765             0.040               10      37   
1  Maricopa  Phoenix  0.015882             0.032               10      30   
2  Maricopa  Phoenix  0.009353             0.016                9      15   
3  Maricopa  Phoenix  0.015882             0.033                9      31   
4  Maricopa  Phoenix  0.007353             0.012                9      11   

   ...  CO 1st Max Hour  CO AQI  SO2 Mean  SO2 1s

Top cities:

In [4]:
city_counts = (
    df_clean.groupby(["State", "City"])
    .size()
    .sort_values(ascending=False)
)

print(city_counts.head(20))

State                 City             
California            Los Angeles          19624
New York              New York             16811
California            Rubidoux             15671
Arizona               Phoenix              14544
Hawaii                Not in a city        12904
Arkansas              North Little Rock    11277
Texas                 Houston              11151
North Carolina        Charlotte            10536
District Of Columbia  Washington           10517
Texas                 El Paso              10366
Utah                  Salt Lake City        9338
Ohio                  Cleveland             8749
California            Concord               8543
                      Vallejo               8453
                      Bethel Island         8432
Arizona               Tucson                8401
California            Capitan               8372
                      Lompoc                8274
Louisiana             Baton Rouge           8254
California            San Pab

real code BEGINS HERE: 

In [5]:
states = ["California", "Pennsylvania", "Texas", "Arizona", "New York"]

for state in states:
    state_data = df_clean[df_clean["State"] == state]

    print(f"\n{state}")
    print(state_data["City"].value_counts().head(10))


California
City
Los Angeles      19624
Rubidoux         15671
Concord           8543
Vallejo           8453
Bethel Island     8432
Capitan           8372
Lompoc            8274
San Pablo         8122
Calexico          8003
Victorville       7836
Name: count, dtype: int64

Pennsylvania
City
Philadelphia     6403
Johnstown        6082
Pittsburgh       4997
Not in a city    4797
York             4479
Charleroi        3995
Bristol          3200
Erie             2969
Scranton         2161
Freemansburg     1918
Name: count, dtype: int64

Texas
City
Houston          11151
El Paso          10366
Dallas            7398
Deer Park         4246
Waco              3542
San Antonio        893
Not in a city      841
Austin             543
Name: count, dtype: int64

Arizona
City
Phoenix       14544
Tucson         8401
Scottsdale     2173
Name: count, dtype: int64

New York
City
New York         16811
Holtsville        4574
Not in a city     1016
Rochester          284
Name: count, dtype: int64


In [6]:
cities = [
    ("California", "Los Angeles"),
    ("Pennsylvania", "Philadelphia"),
    ("Texas", "Houston"),
    ("Arizona", "Phoenix"),
    ("New York", "New York")
] #updated top 5 cities within top 5 states

In [7]:
for state, city in cities:
    subset = df_clean[
        (df_clean["State"] == state) &
        (df_clean["City"] == city)
    ]

    print(f"\n{city}, {state}")
    print("Observations:", len(subset))
    print("Start date:", subset["Date"].min())
    print("End date:", subset["Date"].max())


Los Angeles, California
Observations: 19624
Start date: 2000-03-01
End date: 2023-06-30

Philadelphia, Pennsylvania
Observations: 6403
Start date: 2000-01-01
End date: 2023-06-30

Houston, Texas
Observations: 11151
Start date: 2000-01-01
End date: 2023-06-30

Phoenix, Arizona
Observations: 14544
Start date: 2000-01-01
End date: 2023-09-30

New York, New York
Observations: 16811
Start date: 2000-01-01
End date: 2023-08-31


### Actual Preprocessing

In [8]:
#clean the data and ensure date is correct
df_clean["Date"] = pd.to_datetime(
    df_clean["Date"],
    errors="coerce"
)
print("Invalid dates:", df_clean["Date"].isna().sum())

#remove address and county
df_clean = df_clean.drop(columns=["Address"])
df_clean = df_clean.drop(columns=["County"])


#set timeperiod: 2000 to 2023
start_date = "2000-01-01"
end_date = "2023-06-30"

#clean
df_selected = df_clean[
    (df_clean["Date"] >= start_date) &
    (df_clean["Date"] <= end_date) &
    (
        df_clean[["State", "City"]]
        .apply(tuple, axis=1)
        .isin(cities)
    )
].copy()

Invalid dates: 0


In [9]:
print("Filtered dataset:", df_selected.shape)
print(df_selected.groupby(["State", "City"]).size())

Filtered dataset: (68348, 20)
State         City        
Arizona       Phoenix         14452
California    Los Angeles     19624
New York      New York        16718
Pennsylvania  Philadelphia     6403
Texas         Houston         11151
dtype: int64


In [10]:
#checking for missing/incorrect/duplicate values one more time
missing_values = df_selected.isnull().sum()
print(missing_values[missing_values > 0])

missing_percent = (
    df_selected.isnull().mean() * 100
).sort_values(ascending=False)
print(missing_percent)

print("Duplicate rows:", df_selected.duplicated().sum())

Series([], dtype: int64)
Unnamed: 0           0.0
Date                 0.0
NO2 1st Max Hour     0.0
NO2 1st Max Value    0.0
NO2 Mean             0.0
SO2 AQI              0.0
SO2 1st Max Hour     0.0
SO2 1st Max Value    0.0
SO2 Mean             0.0
CO AQI               0.0
CO 1st Max Hour      0.0
CO 1st Max Value     0.0
CO Mean              0.0
O3 AQI               0.0
O3 1st Max Hour      0.0
O3 1st Max Value     0.0
O3 Mean              0.0
City                 0.0
State                0.0
NO2 AQI              0.0
dtype: float64
Duplicate rows: 0


In [11]:
pollutant_columns = [
    "O3 Mean",
    "CO Mean",
    "SO2 Mean",
    "NO2 Mean",
    "O3 AQI",
    "CO AQI",
    "SO2 AQI",
    "NO2 AQI"
]

for col in pollutant_columns:
    df_selected[col] = pd.to_numeric(
        df_selected[col],
        errors="coerce"
    )
print(df_selected[pollutant_columns].describe())

            O3 Mean       CO Mean      SO2 Mean      NO2 Mean        O3 AQI  \
count  68348.000000  68348.000000  68348.000000  68348.000000  68348.000000   
mean       0.025578      0.418131      1.979001     18.400088     37.277287   
std        0.012082      0.312506      3.216813     10.395482     21.255612   
min        0.000000     -0.400000     -1.725000     -0.152174      0.000000   
25%        0.016353      0.212500      0.237500     10.475000     24.000000   
50%        0.025235      0.333333      0.820833     16.712771     34.000000   
75%        0.034118      0.533333      2.347826     24.791667     44.000000   
max        0.089647      4.246667     53.500000     98.130435    222.000000   

             CO AQI       SO2 AQI       NO2 AQI  
count  68348.000000  68348.000000  68348.000000  
mean       7.046512      6.388439     31.959238  
std        5.828278     11.181706     15.009510  
min        0.000000      0.000000      0.000000  
25%        3.000000      0.000000     